## Evaluate Presidio Analyzer using the Presidio Evaluator framework

This notebook demonstrates how to evaluate a Presidio instance using the presidio-evaluator framework
Steps:
1. Load dataset from file
2. Simple dataset statistics
3. Define the AnalyzerEngine object (and its parameters)
4. Align the dataset's entities to Presidio's entities
5. Set up the Evaluator object
6. Run experiment
7. Evaluate results
8. Error analysis

For an example with a custom Presidio instance, see [notebook 5](5_Evaluate_Custom_Presidio_Analyzer.ipynb).

In [21]:
# install presidio evaluator via pip if not yet installed

#!pip install presidio-evaluator

In [22]:
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError, Plotter
from presidio_evaluator.experiment_tracking import get_experiment_tracker

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2

## 1. Load dataset from file

In [23]:
dataset_name = "generated_size_1500_date_March_15_2026.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd(), "data", dataset_name))
print(len(dataset))

tokenizing input: 100%|██████████| 1500/1500 [00:06<00:00, 229.04it/s]

1500


This dataset was auto generated. See more info here [Synthetic data generation](1_Generate_data.ipynb).

In [24]:
def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter


In [25]:
def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        print("SAMPLE:", sample)
        for tag in sample.tags:
            print("TAG:", tag)
            entity_counter[tag] += 1
    return entity_counter

entity_counts = get_entity_counts(dataset)
print(entity_counts)
pprint(entity_counts.most_common(), compact=True)

SAMPLE: Full text: My birthday is on the weekend. I'll turn 42.
Spans: [Span(type: AGE, value: 42, char_span: [41: 43])]

TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: AGE
TAG: O
SAMPLE: Full text: Mrs. Kreutzmann flew to OSLO on Thursday morning.
Spans: [Span(type: DATE_TIME, value: Thursday, char_span: [32: 40]), Span(type: GPE, value: OSLO, char_span: [24: 28]), Span(type: PERSON, value: Kreutzmann, char_span: [5: 15]), Span(type: TITLE, value: Mrs., char_span: [0: 4])]

TAG: TITLE
TAG: PERSON
TAG: O
TAG: O
TAG: GPE
TAG: O
TAG: DATE_TIME
TAG: O
TAG: O
SAMPLE: Full text: Need to see last 10 transaction of card 4253978024025234633
Spans: [Span(type: CREDIT_CARD, value: 4253978024025234633, char_span: [40: 59])]

TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: CREDIT_CARD
SAMPLE: Full text: You will be responsible for the husbandry and care of a large variety of species including lemurs, antelope, camels, and more
Spans: []

TAG: O
TAG: O
TAG: 

## 2. Simple dataset statistics

In [26]:
entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print("\nMin and max number of tokens in dataset: "\
f"Min: {min([len(sample.tokens) for sample in dataset])}, "\
f"Max: {max([len(sample.tokens) for sample in dataset])}")

print(f"Min and max sentence length in dataset: " \
f"Min: {min([len(sample.full_text) for sample in dataset])}, "\
f"Max: {max([len(sample.full_text) for sample in dataset])}")

print("\nExample InputSample:")
print(dataset[0])

SAMPLE: Full text: My birthday is on the weekend. I'll turn 42.
Spans: [Span(type: AGE, value: 42, char_span: [41: 43])]

TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: AGE
TAG: O
SAMPLE: Full text: Mrs. Kreutzmann flew to OSLO on Thursday morning.
Spans: [Span(type: DATE_TIME, value: Thursday, char_span: [32: 40]), Span(type: GPE, value: OSLO, char_span: [24: 28]), Span(type: PERSON, value: Kreutzmann, char_span: [5: 15]), Span(type: TITLE, value: Mrs., char_span: [0: 4])]

TAG: TITLE
TAG: PERSON
TAG: O
TAG: O
TAG: GPE
TAG: O
TAG: DATE_TIME
TAG: O
TAG: O
SAMPLE: Full text: Need to see last 10 transaction of card 4253978024025234633
Spans: [Span(type: CREDIT_CARD, value: 4253978024025234633, char_span: [40: 59])]

TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: O
TAG: CREDIT_CARD
SAMPLE: Full text: You will be responsible for the husbandry and care of a large variety of species including lemurs, antelope, camels, and more
Spans: []

TAG: O
TAG: O
TAG: 

In [27]:
print("A few examples sentences containing each entity:\n")
for entity in entity_counts.keys():
    samples = [sample for sample in dataset if entity in set(sample.tags)]
    if len(samples) > 1 and entity != "O":
        print(f"Entity: <{entity}> two example sentences:\n"
              f"\n1) {samples[0].full_text}"
              f"\n2) {samples[1].full_text}"
              f"\n------------------------------------\n")

A few examples sentences containing each entity:

Entity: <AGE> two example sentences:

1) My birthday is on the weekend. I'll turn 42.
2) This 73 year old female complaining of stomach pain.
------------------------------------

Entity: <TITLE> two example sentences:

1) Mrs. Kreutzmann flew to OSLO on Thursday morning.
2) Ms. Barese flew to København K on Thursday morning.
------------------------------------

Entity: <PERSON> two example sentences:

1) Mrs. Kreutzmann flew to OSLO on Thursday morning.
2) have you heard Mette Olsen speak yet?
------------------------------------

Entity: <GPE> two example sentences:

1) Mrs. Kreutzmann flew to OSLO on Thursday morning.
2) Iceland was super fun to visit!
------------------------------------

Entity: <DATE_TIME> two example sentences:

1) Mrs. Kreutzmann flew to OSLO on Thursday morning.
2) She was born on 7/8/1978. Her maiden name is Hofman
------------------------------------

Entity: <CREDIT_CARD> two example sentences:

1) Need to 

## 3. Define the AnalyzerEngine object 
Using Presidio with default parameters (not recommended, it's used here for simplicity). For an example on customization, see [notebook 5](5_Evaluate_Custom_Presidio_Analyzer.ipynb)

In [28]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

# Loading the vanilla Analyzer Engine, with the default NER model.
configuration = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
}
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine_with_sm = provider.create_engine()
analyzer_engine = AnalyzerEngine(nlp_engine=nlp_engine_with_sm, default_score_threshold=0.4)

pprint(f"Supported entities for English:")
pprint(analyzer_engine.get_supported_entities("en"), compact=True)

print(f"\nLoaded recognizers for English:")
pprint([rec.name for rec in analyzer_engine.registry.get_recognizers("en", all_fields=True)], compact=True)

print(f"\nLoaded NER models:")
pprint(analyzer_engine.nlp_engine.models)

'Supported entities for English:'
['US_SSN', 'EMAIL', 'URL', 'US_DRIVER_LICENSE', 'AGE', 'US_PASSPORT', 'PERSON',
 'ID', 'MAC_ADDRESS', 'IP_ADDRESS', 'MEDICAL_LICENSE', 'CREDIT_CARD',
 'IBAN_CODE', 'LOCATION', 'DATE_TIME', 'PHONE_NUMBER', 'NRP', 'US_BANK_NUMBER',
 'UK_NHS', 'ORGANIZATION', 'CRYPTO', 'US_ITIN', 'EMAIL_ADDRESS']

Loaded recognizers for English:
['CreditCardRecognizer', 'UsBankRecognizer', 'UsLicenseRecognizer',
 'UsItinRecognizer', 'UsPassportRecognizer', 'UsSsnRecognizer', 'NhsRecognizer',
 'CryptoRecognizer', 'DateRecognizer', 'EmailRecognizer', 'IbanRecognizer',
 'IpRecognizer', 'MedicalLicenseRecognizer', 'MacAddressRecognizer',
 'PhoneRecognizer', 'UrlRecognizer', 'SpacyRecognizer']

Loaded NER models:
[{'lang_code': 'en', 'model_name': 'en_core_web_sm'}]


## 4. Align the dataset's entities to Presidio's entities

There is possibly a difference between the names of entities in the dataset, and the names of entities Presidio can detect.
For example, it could be that a dataset labels a name as PER while Presidio returns PERSON. To be able to compare the predicted value to the actual and gather metrics, an alignment between the entity names is necessary. Consider changing the mapping if your dataset and/or Presidio instance supports difference entity types.

In [29]:
from presidio_evaluator.models import  PresidioAnalyzerWrapper

entities_mapping=PresidioAnalyzerWrapper.presidio_entities_map # default mapping

print("Using this mapping between the dataset and Presidio's entities:")
pprint(entities_mapping, compact=True)


dataset = SpanEvaluator.align_entity_types(
    dataset, 
    entities_mapping=entities_mapping, 
    allow_missing_mappings=True
)
new_entity_counts = get_entity_counts(dataset)
print("\nCount per entity after alignment:")
pprint(new_entity_counts.most_common(), compact=True)

dataset_entities = list(new_entity_counts.values())


Using this mapping between the dataset and Presidio's entities:
{'ADDRESS': 'LOCATION',
 'AGE': 'AGE',
 'BIRTHDAY': 'DATE_TIME',
 'CITY': 'LOCATION',
 'CREDIT_CARD': 'CREDIT_CARD',
 'CREDIT_CARD_NUMBER': 'CREDIT_CARD',
 'DATE': 'DATE_TIME',
 'DATE_OF_BIRTH': 'DATE_TIME',
 'DATE_TIME': 'DATE_TIME',
 'DOB': 'DATE_TIME',
 'DOMAIN': 'URL',
 'DOMAIN_NAME': 'URL',
 'EMAIL': 'EMAIL_ADDRESS',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'FACILITY': 'LOCATION',
 'FIRST_NAME': 'PERSON',
 'GPE': 'LOCATION',
 'HCW': 'PERSON',
 'HOSP': 'ORGANIZATION',
 'HOSPITAL': 'ORGANIZATION',
 'IBAN': 'IBAN_CODE',
 'IBAN_CODE': 'IBAN_CODE',
 'ID': 'ID',
 'IP_ADDRESS': 'IP_ADDRESS',
 'LAST_NAME': 'PERSON',
 'LOC': 'LOCATION',
 'LOCATION': 'LOCATION',
 'NAME': 'PERSON',
 'NATIONALITY': 'NRP',
 'NORP': 'NRP',
 'NRP': 'NRP',
 'O': 'O',
 'ORG': 'ORGANIZATION',
 'ORGANIZATION': 'ORGANIZATION',
 'PATIENT': 'PERSON',
 'PATORG': 'ORGANIZATION',
 'PER': 'PERSON',
 'PERSON': 'PERSON',
 'PHONE': 'PHONE_NUMBER',
 'PHONE_NUMBER': 'PH

## 5. Set up the Evaluator object

In [30]:
# Set up the experiment tracker to log the experiment for reproducibility
experiment = get_experiment_tracker()

# Create the evaluator object
evaluator = SpanEvaluator(model=analyzer_engine)


# Track model and dataset params
params = {"dataset_name": dataset_name, 
          "model_name": evaluator.model.name}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(entities_mapping))

--------
Entities supported by this Presidio Analyzer instance:
US_SSN, EMAIL, URL, US_DRIVER_LICENSE, AGE, US_PASSPORT, PERSON, ID, MAC_ADDRESS, IP_ADDRESS, MEDICAL_LICENSE, CREDIT_CARD, IBAN_CODE, LOCATION, DATE_TIME, PHONE_NUMBER, NRP, US_BANK_NUMBER, UK_NHS, ORGANIZATION, CRYPTO, US_ITIN, EMAIL_ADDRESS


/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/presidio_evaluator/evaluation/base_evaluator.py:83: UserWarning: skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]
  warnings.warn("skip words not provided, using default skip words. "


## 6. Run experiment

In [31]:
%%time

## Run experiment

evaluation_results = evaluator.evaluate_all(dataset)
results = evaluator.calculate_score(evaluation_results)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, 
                                labels=entities)

# end experiment
experiment.end()

Running model PresidioAnalyzerWrapper on dataset...
Finished running model on dataset
saving experiment data to /Users/jyu36/code/ic-llm/presidio/experiment_20260316-194056.json
CPU times: user 9.61 s, sys: 147 ms, total: 9.76 s
Wall time: 10 s


## 7. Evaluate results

In [32]:
# Plot output
plotter = Plotter(results=results, 
                  model_name = evaluator.model.name, 
                  save_as="svg",
                  beta = 2) 

# Path of the directory to save the plots
output_folder = Path(Path.cwd(), "plotter_output")
plotter.plot_scores(output_folder=output_folder)

In [33]:
pprint({"PII F":results.pii_f, "PII recall": results.pii_recall, "PII precision": results.pii_precision})

{'PII F': 0.6608153198897433,
 'PII precision': 0.6938309215536939,
 'PII recall': 0.6530465949820788}


### 7a. False positives
#### Most common false positive tokens:

In [34]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('devil', 18),
 ('iban', 15),
 ('co.ltd', 13),
 ('10th', 10),
 ('pedestrians', 10),
 ('comedy kids family mystery suspense\\ndirected', 9),
 ('zoolander', 8),
 ('hobbits', 8),
 ('fuse tv', 7),
 ('70', 7)]
---------------
Example sentence with each FP token:
	- Devil (`devil` pred as ORGANIZATION)
	- IBAN (`iban` pred as ORGANIZATION)
	- CO.LTD (`co.ltd` pred as O)
	- 10th year (`10th` pred as DATE_TIME)
	- Pedestrians (`pedestrians` pred as NRP)
	- Comedy , Kids & Family Mystery & Suspense\nDirected (`comedy kids family mystery suspense\ndirected` pred as ORGANIZATION)
	- Zoolander (`zoolander` pred as ORGANIZATION)
	- Hobbits (`hobbits` pred as ORGANIZATION)
	- Fuse TV (`fuse tv` pred as ORGANIZATION)
	- 70 year old (`70` pred as O)


[('devil', 18),
 ('iban', 15),
 ('co.ltd', 13),
 ('10th', 10),
 ('pedestrians', 10),
 ('comedy kids family mystery suspense\\ndirected', 9),
 ('zoolander', 8),
 ('hobbits', 8),
 ('fuse tv', 7),
 ('70', 7)]

#### More FP analysis

In [35]:
fps_df = ModelError.get_fps_dataframe(results.model_errors, entity=["PERSON"])
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,Discipline,discipline,O,PERSON
1,Salesperson,salesperson,O,PERSON
2,Orchestra,orchestra,O,PERSON
3,Salesperson,salesperson,O,PERSON
4,Discipline,discipline,O,PERSON
5,Orchestra,orchestra,O,PERSON
6,Discipline,discipline,O,PERSON
7,Discipline,discipline,O,PERSON
8,Salesperson,salesperson,O,PERSON
9,Discipline,discipline,O,PERSON


### 7b. False negatives (FN)

#### Most common false negative examples + a few samples with FN

In [36]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

Most common false negative tokens:
[('ms.', 10),
 ('dr.', 8),
 ('38', 7),
 ('19', 7),
 ('73', 6),
 ('70', 6),
 ('cyprus greek', 5),
 ('53', 5),
 ('greenlander', 5),
 ('55', 4),
 ('77', 4),
 ('43', 4),
 ('72', 4),
 ('62', 4),
 ('76', 4)]
---------------
Example sentence with each FN token:
	- Ms. (`ms.` annotated as O)
	- Dr. (`dr.` annotated as O)
	- 38 (`38` annotated as O)
	- 19 (`19` annotated as O)
	- 73 (`73` annotated as O)
	- 70 (`70` annotated as O)
	- Cyprus ( Greek ) (`cyprus greek` annotated as O)
	- 53 (`53` annotated as O)
	- Greenlander (`greenlander` annotated as O)
	- 55 (`55` annotated as O)
	- 77 (`77` annotated as O)
	- 43 (`43` annotated as O)
	- 72 (`72` annotated as O)
	- 62 (`62` annotated as O)
	- 76 (`76` annotated as O)


[('ms.', 10),
 ('dr.', 8),
 ('38', 7),
 ('19', 7),
 ('73', 6),
 ('70', 6),
 ('cyprus greek', 5),
 ('53', 5),
 ('greenlander', 5),
 ('55', 4),
 ('77', 4),
 ('43', 4),
 ('72', 4),
 ('62', 4),
 ('76', 4)]

#### More FN analysis

In [37]:
fns_df = ModelError.get_fns_dataframe(results.model_errors)

In [38]:
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,42,42,O,AGE
1,OSLO,oslo,O,LOCATION
2,4253978024025234633,4253978024025234633,O,CREDIT_CARD
3,Bedřicha Smetany 405 Kutná Hora 1 Stipo Way,bedřicha smetany 405 kutná hora 1 stipo way,O,LOCATION
4,314 Αρτεμισίου 62 Apt . 601 ΛΕΜΕΣΟΣ LI 72513 Ibirapita 8057 Apt . 286 Dolores Uruguay,314 αρτεμισίου 62 601 λεμεσος li 72513 ibirapita 8057 286 dolores uruguay,O,LOCATION
5,73,73,O,AGE
6,Hofman,hofman,O,PERSON
7,AK Steel Holding Corporation\n,ak steel holding corporation\n,O,ORGANIZATION
8,489 Slovenčeva 107\n Suite 385\n Dutovlje\n Slovenia 08503,489 slovenčeva 107\n 385\n dutovlje\n slovenia 08503,O,LOCATION
9,9865 2117,9865 2117,O,PHONE_NUMBER
